### P-center problem

The first mixed-integer programming (MIP) formulation for the discrete $p$-center problem was proposed by Daskin (2013). The following decision variables are defined:  
$y_i = 1$ if a facility is located at node $i \in I$, and $0$ otherwise;  
$x_{ij} = 1$ if customer $j \in J$ is assigned to a facility located at node $i \in I$, and $0$ otherwise.  

Daskin’s formulation can be expressed as follows:

$$
\begin{aligned}
\text{Min} \quad & z \\
\text{subject to} \quad
& \sum_{i \in I} d_{ji} x_{ij} \leq z, \quad \forall j \in J, \\
& \sum_{i \in I} x_{ij} = 1, \quad \forall j \in J, \\
& x_{ij} \leq y_i, \quad \forall i \in I,\; j \in J, \\
& \sum_{i \in I} y_i \leq p, \\
& y_i \in \{0,1\}, \quad \forall i \in I, \\
& x_{ij} \in \{0,1\}, \quad \forall i \in I,\; j \in J.
\end{aligned}
$$

In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Number of candidate facility sites
num_facilities = 5

# Number of customers
num_customers = 5

# Sets
I = range(num_facilities)
J = range(num_customers)

# Symmetric service cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Customer demand
d = np.array([10, 8, 5, 8, 12])

# Number of facilities to locate
p = 2

In [ ]:
m = new_ampl()
m.eval(r"""
set I;
set J;
param c {I,J};
param d {J} >= 0;
var x {I,J} binary;
var y {I} binary;
subject to Assignment {j in J}: sum {i in I} x[i,j] = 1;
param p integer >= 1;
param exact_count binary default 0;
var z >= 0;
minimize Total_Cost: z;
subject to CardinalityUpper: sum {i in I} y[i] <= p;
subject to CardinalityLower: sum {i in I} y[i] >= p*exact_count;
subject to Link {i in I,j in J}: x[i,j] <= y[i];
subject to Radius {j in J}: sum {i in I} c[i,j]*x[i,j] <= z;
""")
m.set["I"] = list(I)
m.set["J"] = list(J)
m.param["c"] = {(i,j): float(c[i,j]) for i in I for j in J}
m.param["d"] = {j: float(d[j]) for j in J}
m.param["p"] = p

solve_checked(m)
x = values(m, "x")
y = values(m, "y")
z = m.var["z"].value()


In [ ]:
if m.solve_result == "solved":
    print("Optimal solution")
    print("Total cost:", m.obj["Total_Cost"].value())
    for i in I:
        if y[i] > 0.1:
            print("Open facility:", i)
else:
    print("Solver status:", m.solve_result)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle



def solve_and_plot_p_center(num_facilities=5, num_customers=5, p=2, seed=42,
                            coord_range=(0, 100), metric="euclidean"):
    rng = np.random.default_rng(seed)

    I = range(num_facilities)
    J = range(num_customers)

    facilities_xy = rng.uniform(coord_range[0], coord_range[1], size=(num_facilities, 2))
    customers_xy = rng.uniform(coord_range[0], coord_range[1], size=(num_customers, 2))

    if metric.lower() == "euclidean":
        diff = facilities_xy[:, None, :] - customers_xy[None, :, :]
        c = np.sqrt((diff ** 2).sum(axis=2))
    elif metric.lower() == "manhattan":
        diff = np.abs(facilities_xy[:, None, :] - customers_xy[None, :, :])
        c = diff.sum(axis=2)
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    m = new_ampl()
    m.eval(r"""
    set I;
    set J;
    param c {I,J};
    param d {J} >= 0;
    var x {I,J} binary;
    var y {I} binary;
    subject to Assignment {j in J}: sum {i in I} x[i,j] = 1;
    param p integer >= 1;
    param exact_count binary default 0;
    var z >= 0;
    minimize Total_Cost: z;
    subject to CardinalityUpper: sum {i in I} y[i] <= p;
    subject to CardinalityLower: sum {i in I} y[i] >= p*exact_count;
    subject to Link {i in I,j in J}: x[i,j] <= y[i];
    subject to Radius {j in J}: sum {i in I} c[i,j]*x[i,j] <= z;
    """)
    m.set["I"] = list(I)
    m.set["J"] = list(J)
    m.param["c"] = {(i,j): float(c[i,j]) for i in I for j in J}
    m.param["d"] = {j: 1.0 for j in J}
    m.param["p"] = p
    m.param["exact_count"] = 1
    
    solve_checked(m)
    x = values(m, "x")
    y = values(m, "y")
    z = m.var["z"].value()

    open_facilities = [i for i in I if y[i] > 0.5]
    assignments = {}  # j -> i
    total_distance = 0.0
    min_dist = float("inf")
    max_dist = 0.0

    for j in J:
        i_star = None
        for i in I:
            if x[i, j] > 0.5:
                i_star = i
                break
        assignments[j] = i_star
        dist = float(c[i_star, j])
        total_distance += dist
        min_dist = min(min_dist, dist)
        max_dist = max(max_dist, dist)

    print(f"Total distance traveled by all customers: {total_distance:.4f}")
    print(f"Minimum distance traveled by a customer: {min_dist:.4f}")
    print(f"Maximum assigned distance (z): {z:.4f}")
    for i in open_facilities:
        print("Open facility at:", i)

    fig, ax = plt.subplots(figsize=(9, 7))

    # Customers
    ax.scatter(customers_xy[:, 0], customers_xy[:, 1], marker="o")
    for j in J:
        ax.annotate(f"C{j}", (customers_xy[j, 0], customers_xy[j, 1]),
                    xytext=(5, 5), textcoords="offset points")

    # Closed facilities
    closed_facilities = [i for i in I if i not in open_facilities]
    if closed_facilities:
        ax.scatter(facilities_xy[closed_facilities, 0], facilities_xy[closed_facilities, 1], marker="s")
        for i in closed_facilities:
            ax.annotate(f"F{i}", (facilities_xy[i, 0], facilities_xy[i, 1]),
                        xytext=(5, 5), textcoords="offset points")

    # Open facilities (highlighted) + circles with radius z
    if open_facilities:
        ax.scatter(facilities_xy[open_facilities, 0], facilities_xy[open_facilities, 1], marker="s")
        for i in open_facilities:
            ax.annotate(f"F{i}*", (facilities_xy[i, 0], facilities_xy[i, 1]),
                        xytext=(5, 5), textcoords="offset points")
            circ = Circle((facilities_xy[i, 0], facilities_xy[i, 1]), radius=float(z),
                          fill=False, linewidth=1.5)
            ax.add_patch(circ)

    # Assignment lines
    for j, i in assignments.items():
        x0, y0 = customers_xy[j, 0], customers_xy[j, 1]
        x1, y1 = facilities_xy[i, 0], facilities_xy[i, 1]
        ax.plot([x0, x1], [y0, y1], linewidth=1)

    ax.set_title(f"p-center solution (p={p}) - z={z:.4f} - metric: {metric}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.axis("equal")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return m



In [ ]:
solve_and_plot_p_center(num_facilities=15, num_customers=20, p = 4 ,seed=1234, metric="euclidean")